# Cost Matrix — Definition & Visualization

The cost matrix `C` of shape `(N, N)` is the backbone of the warehouse problem.  
`C[i, j]` = cost of shipping one unit from warehouse `i` to warehouse `j`.

**Properties enforced by the environment:**
- `C[i, i] = 0` — no cost to stay
- `C[i, j] = C[j, i]` — symmetric (same cost both ways)
- Values in `[0.5, 2.0]` when randomly generated

This notebook lets you:
1. Define a cost matrix manually or generate one randomly
2. Visualize it as a **heatmap**
3. Visualize it as a **warehouse network graph** (nodes = warehouses, edge thickness = cost)
4. Plug it back into the environment

In [ ]:
import sys
sys.path.append('..')  # so we can import from the project root

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'monospace',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

WAREHOUSE_NAMES = [f'W{i}' for i in range(1, 17)]  # up to 16 warehouses
SEED = 42

---
## 1 — Define the Cost Matrix

Choose **one** of the three cells below. Comment out the others.

In [ ]:
# ── OPTION A: Manual definition (small scenario, N=4) ──────────────────────
# Rows/cols: W1, W2, W3, W4
# Story: W1-W2 are close (cost 0.5), W3-W4 are far from everyone else (cost 1.8+)

C_manual = np.array([
    #  W1    W2    W3    W4
    [0.0,  0.5,  1.2,  1.8],  # from W1
    [0.5,  0.0,  1.5,  1.9],  # from W2
    [1.2,  1.5,  0.0,  0.7],  # from W3
    [1.8,  1.9,  0.7,  0.0],  # from W4
], dtype=np.float32)

N = 4
C = C_manual
labels = WAREHOUSE_NAMES[:N]
print(f'Cost matrix (N={N}):')
print(C)

In [ ]:
# ── OPTION B: Random generation (mirrors the WarehouseEnv logic) ───────────
# Uncomment this cell to use it

# N = 9  # try 4, 9, or 16
# rng = np.random.default_rng(SEED)
# c = rng.uniform(0.5, 2.0, size=(N, N)).astype(np.float32)
# c = (c + c.T) / 2           # symmetrise
# np.fill_diagonal(c, 0.0)    # zero diagonal
# C = c
# labels = WAREHOUSE_NAMES[:N]
# print(f'Random cost matrix (N={N}, seed={SEED}):')
# print(C.round(2))

In [ ]:
# ── OPTION C: Load from env (uses config) ─────────────────────────────────
# Uncomment to pull the exact matrix the env will use

# import yaml
# from env.warehouse_env import WarehouseEnv
# with open('../configs/small.yaml') as f:
#     cfg_override = yaml.safe_load(f)
# with open('../configs/default.yaml') as f:
#     cfg = yaml.safe_load(f)
# cfg['env'].update(cfg_override.get('env', {}))
# env = WarehouseEnv(cfg['env'], seed=SEED)
# C = env.cost_matrix
# N = env.n
# labels = WAREHOUSE_NAMES[:N]

---
## 2 — Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(max(5, N * 0.9), max(4, N * 0.8)))

# Mask the diagonal (self-shipment = 0, not interesting)
mask = np.eye(N, dtype=bool)

sns.heatmap(
    C,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='#2c2c2c',
    xticklabels=labels,
    yticklabels=labels,
    cbar_kws={'label': 'Shipping cost per unit', 'shrink': 0.8},
    ax=ax,
    vmin=0,
    vmax=C.max(),
)

# Draw diagonal as grey boxes
for i in range(N):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color='#cccccc', lw=0))
    ax.text(i + 0.5, i + 0.5, '—', ha='center', va='center', fontsize=11, color='#888')

ax.set_title(f'Shipping Cost Matrix  (N={N} warehouses)', fontsize=14, pad=14, fontweight='bold')
ax.set_xlabel('Destination warehouse', labelpad=8)
ax.set_ylabel('Source warehouse', labelpad=8)
ax.tick_params(left=False, bottom=False)

plt.tight_layout()
plt.savefig('../outputs/cost_matrix_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/cost_matrix_heatmap.png')

---
## 3 — Warehouse Network Graph

Each warehouse is a node. Each edge represents a shipping route.  
**Edge thickness and color** encode the shipping cost — thick dark red = expensive, thin light = cheap.

In [ ]:
def draw_warehouse_network(C, labels, title='Warehouse Shipping Network', ax=None):
    N = len(labels)
    G = nx.Graph()
    G.add_nodes_from(range(N))

    edges, weights = [], []
    for i in range(N):
        for j in range(i + 1, N):
            G.add_edge(i, j, weight=C[i, j])
            edges.append((i, j))
            weights.append(C[i, j])

    # Layout: circular for small N, spring for larger
    pos = nx.circular_layout(G) if N <= 6 else nx.kamada_kawai_layout(G, weight='weight')

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))
    ax.set_facecolor('#0f0f1a')

    # Normalise weights for visual encoding
    w_arr = np.array(weights)
    w_norm = (w_arr - w_arr.min()) / (w_arr.max() - w_arr.min() + 1e-8)
    edge_widths = 1.0 + w_norm * 6.0   # 1px (cheap) → 7px (expensive)

    cmap = plt.cm.YlOrRd
    edge_colors = [cmap(v) for v in w_norm]

    # Draw edges
    nx.draw_networkx_edges(
        G, pos, edgelist=edges,
        width=edge_widths,
        edge_color=edge_colors,
        ax=ax, alpha=0.85,
    )

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos,
        node_size=900,
        node_color='#4fc3f7',
        edgecolors='white',
        linewidths=2,
        ax=ax,
    )

    # Node labels
    nx.draw_networkx_labels(
        G, pos,
        labels={i: labels[i] for i in range(N)},
        font_color='#0f0f1a',
        font_weight='bold',
        font_size=10,
        ax=ax,
    )

    # Edge weight annotations (only for small N)
    if N <= 6:
        edge_labels = {(i, j): f'{C[i,j]:.1f}' for i, j in edges}
        nx.draw_networkx_edge_labels(
            G, pos, edge_labels=edge_labels,
            font_size=8, font_color='white',
            bbox=dict(boxstyle='round,pad=0.2', fc='#1a1a2e', alpha=0.7, ec='none'),
            ax=ax,
        )

    # Colorbar legend
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(w_arr.min(), w_arr.max()))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Shipping cost', shrink=0.7, pad=0.02)

    ax.set_title(title, fontsize=13, fontweight='bold', color='white', pad=12)
    ax.axis('off')
    return ax


fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('#0f0f1a')
draw_warehouse_network(C, labels, title=f'Warehouse Shipping Network  (N={N})', ax=ax)
plt.tight_layout()
plt.savefig('../outputs/cost_matrix_network.png', bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved → outputs/cost_matrix_network.png')

---
## 4 — Side-by-Side: Cheap vs Expensive Network Structures

Compare two extreme cost regimes to build intuition about how structure affects the agent's strategy.

In [ ]:
def make_hub_spoke(n, hub_cost=0.3, spoke_cost=1.9, seed=0):
    """Hub-and-spoke: warehouse 0 is cheap to reach from everywhere. Rest are expensive."""
    rng = np.random.default_rng(seed)
    C = rng.uniform(spoke_cost * 0.8, spoke_cost, size=(n, n)).astype(np.float32)
    C = (C + C.T) / 2
    # Make hub (node 0) cheap
    C[0, :] = hub_cost
    C[:, 0] = hub_cost
    np.fill_diagonal(C, 0.0)
    return C

def make_uniform(n, cost=1.0):
    """All routes cost the same — agent must rely purely on demand signal."""
    C = np.full((n, n), cost, dtype=np.float32)
    np.fill_diagonal(C, 0.0)
    return C


n_compare = 5
lbs = WAREHOUSE_NAMES[:n_compare]

C_hub     = make_hub_spoke(n_compare)
C_uniform = make_uniform(n_compare)
C_random  = (lambda rng: (lambda c: (np.fill_diagonal(c, 0) or c))(
                (lambda c: (c + c.T) / 2)(rng.uniform(0.5, 2.0, (n_compare, n_compare)).astype(np.float32))
             ))(np.random.default_rng(SEED))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0f0f1a')

draw_warehouse_network(C_hub,     lbs, title='Hub-and-Spoke\n(W1 is the cheap hub)',  ax=axes[0])
draw_warehouse_network(C_uniform, lbs, title='Uniform Costs\n(all routes equal)',      ax=axes[1])
draw_warehouse_network(C_random,  lbs, title='Random Costs\n(our default setting)',    ax=axes[2])

fig.suptitle('Cost Matrix Structures — Effect on Agent Strategy', 
             fontsize=15, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/cost_matrix_comparison.png', bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved → outputs/cost_matrix_comparison.png')

---
## 5 — Plug Your Matrix Back into the Environment

Pass `cost_matrix` in the config to override random generation.

In [ ]:
import sys, yaml
sys.path.append('..')
from env.warehouse_env import WarehouseEnv

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# Inject our manual cost matrix into the env config
cfg['env']['n_warehouses'] = N
cfg['env']['cost_matrix'] = C.tolist()   # YAML needs a plain list

env = WarehouseEnv(cfg['env'], seed=SEED)
state = env.reset()

print('✓ Environment created with custom cost matrix')
print(f'  n_warehouses  : {env.n}')
print(f'  inventory     : {state["inventory"].round(1)}')
print(f'  demand        : {state["demand"].round(1)}')
print(f'  cost_matrix OK: {np.allclose(env.cost_matrix, C)}')

# One random step to confirm step() works
action = env.sample_action()
next_state, reward, done, info = env.step(action)
print(f'\nOne random step:')
print(f'  reward           : {reward:.3f}')
print(f'  transport_cost   : {info["transport_cost"]:.3f}')
print(f'  unmet_demand     : {info["unmet_demand"]:.3f}')
print(f'  demand_satisf.   : {info["demand_satisfaction"]:.3f}')